In [ ]:
# --- Environment setup: run this cell first (Colab or local) -------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cto-school/agentic-ai-engineering.git"   # the public course repository
REPO_DIR = Path("/content/agentic-ai-engineering")

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        print("Installing requirements (this takes a minute) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-core.txt")], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-day2.txt")], check=True)
    os.chdir(REPO_DIR / "day_02_knowledge_and_state")
    # Colab has no .env file. Paste the key you were issued; it is kept only in this runtime.
    from getpass import getpass
    if not os.getenv("OPENROUTER_API_KEY"):
        key = getpass("OPENROUTER_API_KEY (press Enter to stay in mock mode): ").strip()
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
else:
    # Local machine: the key is read from the .env file at the repository root
    # (Day 1.1 explains how to create it from .env.example).
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))

print("Working directory:", os.getcwd())
print("Mode:", "LIVE (OpenRouter key found)" if os.getenv("OPENROUTER_API_KEY") else "MOCK (no key found: deterministic answers, no credit spent)")


# Day 2.1 — Documents and Chunks
The model has never seen our fictional campus documents. Before it can answer questions
about them, we have to turn files into small, labelled pieces a retriever can rank.

```text
Markdown files -> sections -> chunks with source, section and a stable id
```


## Before you begin

### Learning outcomes

- Turn three Markdown documents into retrieval units that keep their source and section.
- See why sending whole documents, or cutting them into blind character slices, both fail.

Architecture reference: [D06](../diagrams/source/day_02.md).

### Expected observation

Fifteen chunks, each printed with a stable id such as `battery_safety:data-retention`,
its source file, its section heading and its length.

### API key reminder

Day 2 needs **no API key** until notebook 04, and even there a deterministic offline
generator takes over automatically. If you do want live answers later, create the `.env`
file exactly as described in **Day 1.1 — First Model Call** (repository root, one line
`OPENROUTER_API_KEY=sk-or-...`). Day 1 owns that guide; nothing here repeats it.

## Concept briefing

## Why retrieval is an application problem

A model may know general facts, but a course application often needs supplied manuals,
project documents or current organisational information. Placing every document in every
request is expensive, noisy and eventually impossible. Retrieval selects a small amount
of evidence relevant to the current question and places it into the model context.

Retrieval-Augmented Generation is therefore a pipeline, not a model feature:

```text
documents -> chunks -> representations -> index
question -> retrieval -> selected evidence -> generation -> validation
```

Every arrow can fail. Debugging RAG requires identifying which arrow failed rather than
changing prompts at random.

## Why documents become chunks

Retrieval operates on units. A whole manual may contain the answer but also thousands of
irrelevant words. A tiny fragment may match a keyword but lack the surrounding condition
that changes its meaning. Chunking balances retrieval precision against sufficient
context.

Useful chunks retain provenance: source file, section heading, stable identifier and
text. Without this metadata the application cannot cite the result, evaluate expected
sections, or explain why a passage was retrieved.

There is no universal chunk size. Structure-aware chunks are often easier to inspect than
blind character windows for small engineering documents. The course therefore starts
with headings rather than presenting chunking as an arbitrary numeric tuning exercise.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/knowledge_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 2 keeps all documents in data/corpus. Everything we index comes from there.
CORPUS_DIR = PROJECT_ROOT / "data" / "corpus"
print("Corpus folder:", CORPUS_DIR)

## Step 1 — Look at the raw documents first

Never index something you have not read. Three small Markdown files stand in for the
manuals of a campus microgrid. Small is deliberate: you can check every retrieval result
by hand, which is impossible with a real 400-page manual.

In [ ]:
# Sort so the order is the same on every machine.
files = sorted(CORPUS_DIR.glob("*.md"))
print("Documents found:", len(files))

for path in files:
    text = path.read_text(encoding="utf-8")
    first_line = text.splitlines()[0]            # the "# Title" line
    headings = [line for line in text.splitlines() if line.startswith("## ")]
    print()
    print("file      :", path.name)
    print("size      :", path.stat().st_size, "bytes")
    print("title     :", first_line.lstrip("# "))
    print("sections  :", len(headings), "->", [h[3:] for h in headings])

## Step 2 — Why not just send the whole corpus?

The obvious idea is to paste all three documents into every request. Measure it first:
context costs money on every call, and a real manual is a hundred times bigger.

In [ ]:
whole_corpus = "\n".join(path.read_text(encoding="utf-8") for path in files)
words = len(whole_corpus.split())

# A rough rule of thumb: English text is about 3/4 of a word per token.
approx_tokens = int(words / 0.75)

print("Whole corpus     :", words, "words  (~", approx_tokens, "tokens)")
print("A typical question:", len("How long are battery fault records retained?".split()), "words")
print()
print("So >99% of the tokens we would send are irrelevant to any one question,")
print("and they are paid for on every single call.")
print("Retrieval's job: send only the few hundred words that matter.")

## Step 3 — Split by heading into chunks

`load_markdown_corpus` walks each file and starts a new chunk at every `##` heading. Each
chunk keeps the file name, the document title, the section heading, and an id built from
both (`file_stem:section-slug`). That id is what a citation will point at later.

In [ ]:
from knowledge_agent.documents import load_markdown_corpus

chunks = load_markdown_corpus(CORPUS_DIR)
print("Chunks created:", len(chunks))
print()

# One line per chunk: id, source file, section heading, size in characters.
print(f"{'chunk_id':42}{'source':24}{'section':28}chars")
print("-" * 100)
for chunk in chunks:
    print(f"{chunk.chunk_id:42}{chunk.source:24}{chunk.section:28}{len(chunk.text)}")

## Step 4 — Inspect one chunk completely

A chunk is a small object, not a string. Print every field of the chunk that holds the
five-minute reconnection rule; these are exactly the fields a citation will quote.

In [ ]:
# next(...) returns the first chunk whose text contains the phrase.
target = next(chunk for chunk in chunks if "five continuous minutes" in chunk.text)

print("chunk_id :", target.chunk_id)
print("source   :", target.source)
print("title    :", target.title)
print("section  :", target.section)
print("text     :")
print(target.text)
print()
print("searchable_text is what the retriever will index (title + section + text):")
print(repr(target.searchable_text[:120]) + " ...")

## Step 5 — Break it: blind character slices

Chunking is not "cut every N characters". Slice the same file every 250 characters and
compare: the slices have no source, no section and no id, and one of them cuts a rule in
half so neither half can answer the question on its own.

In [ ]:
raw = (CORPUS_DIR / "solar_microgrid.md").read_text(encoding="utf-8")
SLICE = 250
slices = [raw[start:start + SLICE] for start in range(0, len(raw), SLICE)]
print("Blind slices:", len(slices), "- each one carries no source, no section, no id.")
print()

rule = "five continuous minutes"
for number, piece in enumerate(slices, start=1):
    if rule in piece:
        print(f"slice {number} starts: ...{piece[:70]!r}")
        print(f"slice {number} ends  : {piece[-70:]!r}...")

sentence = "utility voltage, frequency, and phase remain within synchronization limits for five continuous minutes"
print()
print("Whole rule inside ONE blind slice   :", any(sentence in piece for piece in slices))
print("Whole rule inside ONE heading chunk :", any(sentence in chunk.text for chunk in chunks))

### Try it yourself

The battery guide mentions `45 °C`. Predict how many chunks contain that string, then run
the cell. Would retrieving *any* of them answer "at what temperature does charging stop?"

In [ ]:
# --- Worked solution ---
needle = "45 °C"
matches = [chunk for chunk in chunks if needle in chunk.text]

print("Chunks containing", needle, ":", len(matches))
for chunk in matches:
    print()
    print(" ", chunk.chunk_id, "|", chunk.section)
    print("  ", chunk.text[:150], "...")

print()
print("Both sections mention 45 degrees, but only 'Warning and shutdown' says what happens")
print("at that temperature. Retrieval must pick the right SECTION, not just the right file -")
print("which is why we evaluate section hits, not only source hits, in notebook 06.")

### Checkpoint

**1. Why does a chunk store source and section instead of only text?**

<details><summary>Show answer</summary>

Because everything downstream needs provenance: a citation must name a file and a section,
evaluation compares the retrieved section with the expected one, and a debugging session
asks "which document did this sentence come from?". Text alone cannot answer any of those,
and the model cannot invent the metadata reliably.

</details>

**2. We produced 15 chunks. Why not 3 chunks, one per document?**

<details><summary>Show answer</summary>

A whole document usually matches every question a little and no question precisely, so
ranking becomes meaningless and the model receives thousands of irrelevant words. Going
too far the other way (one sentence per chunk) loses the condition that gives a sentence
its meaning. Heading sections are a good middle for structured engineering documents.

</details>

### Recap

- **Limitation we saw:** whole documents are too big to send, and blind character slices
  destroy both meaning and metadata.
- **Layer we added:** heading-aware chunking that keeps `source`, `section` and a stable
  `chunk_id`.
- **Evidence it worked:** 15 chunks printed with ids, and the five-minute reconnection
  rule survives intact inside a single chunk.